# Qwen2-VL-2B Calorie Estimator on Nutrition5k
Run all cells to perform model inference across the 219 plate images.

In [ ]:
!pip install -q transformers torch torchvision accelerate pillow pandas tqdm requests

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

model_name = 'Qwen/Qwen2-VL-2B-Instruct'
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
)
processor = AutoProcessor.from_pretrained(model_name)

In [ ]:
import pandas as pd, json
records = json.loads('[{"dish_id": "dish_1559844951", "tier": "Low", "ground_truth_calories": 274.3, "total_mass_grams": 122.0, "total_fat_grams": 17.9, "total_carb_grams": 22.8, "total_protein_grams": 9.8, "num_ingredients": 2, "ingredient_summary": "almonds (36.0g), potatoes (86.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559844951/rgb.png"}, {"dish_id": "dish_1561575474", "tier": "Low", "ground_truth_calories": 120.1, "total_mass_grams": 183.0, "total_fat_grams": 5.0, "total_carb_grams": 17.4, "total_protein_grams": 3.0, "num_ingredients": 5, "ingredient_summary": "salt (0.6g), eggplant (57.8g), roasted potatoes (60.0g), cauliflower (64.0g), olive oil (0.6g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561575474/rgb.png"}, {"dish_id": "dish_1563216717", "tier": "Low", "ground_truth_calories": 246.0, "total_mass_grams": 332.0, "total_fat_grams": 5.6, "total_carb_grams": 16.3, "total_protein_grams": 33.1, "num_ingredients": 12, "ingredient_summary": "garlic (0.9g), squash (63.1g), cherry tomatoes (4.5g), chicken breast (89.5g), spinach (raw) (2.2g), parmesan cheese (2.2g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563216717/rgb.png"}, {"dish_id": "dish_1561662216", "tier": "Low", "ground_truth_calories": 300.8, "total_mass_grams": 193.0, "total_fat_grams": 12.4, "total_carb_grams": 28.2, "total_protein_grams": 18.6, "num_ingredients": 17, "ingredient_summary": "soy sauce (3.4g), garlic (2.1g), white rice (8.5g), parsley (0.2g), onions (1.7g), brown rice (68.0g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561662216/rgb.png"}, {"dish_id": "dish_1563207364", "tier": "Low", "ground_truth_calories": 309.3, "total_mass_grams": 271.0, "total_fat_grams": 13.8, "total_carb_grams": 30.7, "total_protein_grams": 15.0, "num_ingredients": 6, "ingredient_summary": "scrambled eggs (68.0g), yam (50.0g), grapes (30.0g), egg whites (67.0g), olive oil (6.0g), sweet potato (50.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563207364/rgb.png"}, {"dish_id": "dish_1561739238", "tier": "Low", "ground_truth_calories": 274.2, "total_mass_grams": 257.0, "total_fat_grams": 9.8, "total_carb_grams": 37.6, "total_protein_grams": 9.5, "num_ingredients": 4, "ingredient_summary": "egg whites (71.0g), hash browns (43.0g), berries (61.0g), sweet potato (82.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561739238/rgb.png"}, {"dish_id": "dish_1562611680", "tier": "Low", "ground_truth_calories": 326.9, "total_mass_grams": 256.0, "total_fat_grams": 19.4, "total_carb_grams": 24.8, "total_protein_grams": 16.0, "num_ingredients": 18, "ingredient_summary": "cilantro (0.1g), garlic (0.4g), olive oil (7.1g), pepperoni pizza (50.0g), lemon juice (0.1g), onions (1.6g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562611680/rgb.png"}, {"dish_id": "dish_1560453174", "tier": "Low", "ground_truth_calories": 329.7, "total_mass_grams": 256.0, "total_fat_grams": 18.1, "total_carb_grams": 10.5, "total_protein_grams": 32.1, "num_ingredients": 20, "ingredient_summary": "basil (0.2g), rosemary (0.2g), carrot (5.6g), olive oil (13.0g), almonds (2.8g), lemon juice (0.5g), +14 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560453174/rgb.png"}, {"dish_id": "dish_1560543755", "tier": "Low", "ground_truth_calories": 257.9, "total_mass_grams": 264.0, "total_fat_grams": 8.9, "total_carb_grams": 27.4, "total_protein_grams": 17.4, "num_ingredients": 19, "ingredient_summary": "green onions (1.2g), chicken (43.0g), cauliflower (75.0g), green beans (1.1g), spinach (raw) (1.1g), ginger (0.2g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560543755/rgb.png"}, {"dish_id": "dish_1563996485", "tier": "Low", "ground_truth_calories": 178.0, "total_mass_grams": 201.0, "total_fat_grams": 11.6, "total_carb_grams": 16.6, "total_protein_grams": 4.2, "num_ingredients": 11, "ingredient_summary": "salt (0.3g), pepper (0.0g), blueberries (3.1g), chard (7.8g), arugula (43.9g), vinegar (1.5g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563996485/rgb.png"}, {"dish_id": "dish_1564761488", "tier": "Low", "ground_truth_calories": 328.0, "total_mass_grams": 336.0, "total_fat_grams": 8.4, "total_carb_grams": 40.6, "total_protein_grams": 21.7, "num_ingredients": 4, "ingredient_summary": "turkey bacon (21.0g), egg whites (105.0g), fried rice (97.0g), honeydew melons (113.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564761488/rgb.png"}, {"dish_id": "dish_1559678330", "tier": "Low", "ground_truth_calories": 169.7, "total_mass_grams": 150.0, "total_fat_grams": 16.2, "total_carb_grams": 5.3, "total_protein_grams": 3.2, "num_ingredients": 9, "ingredient_summary": "olive oil (15.7g), spinach (raw) (78.4g), vinegar (7.8g), tomatoes (9.8g), garlic (1.5g), cucumbers (15.7g), +3 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559678330/rgb.png"}, {"dish_id": "dish_1564428444", "tier": "Low", "ground_truth_calories": 157.0, "total_mass_grams": 277.0, "total_fat_grams": 10.3, "total_carb_grams": 14.9, "total_protein_grams": 4.7, "num_ingredients": 10, "ingredient_summary": "olive oil (9.5g), onions (7.1g), salt (0.3g), garlic (0.9g), pears (11.9g), mixed greens (59.4g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564428444/rgb.png"}, {"dish_id": "dish_1558459115", "tier": "Low", "ground_truth_calories": 271.5, "total_mass_grams": 298.0, "total_fat_grams": 10.5, "total_carb_grams": 43.5, "total_protein_grams": 6.6, "num_ingredients": 5, "ingredient_summary": "almonds (20.0g), apple (143.0g), white rice (32.0g), cherry tomatoes (61.0g), grapes (42.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558459115/rgb.png"}, {"dish_id": "dish_1563998210", "tier": "Low", "ground_truth_calories": 148.1, "total_mass_grams": 91.0, "total_fat_grams": 5.0, "total_carb_grams": 2.2, "total_protein_grams": 22.6, "num_ingredients": 17, "ingredient_summary": "garlic (0.7g), olive oil (0.7g), thyme (0.2g), lemon juice (0.2g), butter (0.7g), cream (2.8g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563998210/rgb.png"}, {"dish_id": "dish_1564777960", "tier": "Low", "ground_truth_calories": 331.0, "total_mass_grams": 147.0, "total_fat_grams": 16.7, "total_carb_grams": 16.3, "total_protein_grams": 26.4, "num_ingredients": 10, "ingredient_summary": "spinach (cooked) (2.7g), cheese pizza (39.0g), shallots (5.5g), onions (2.7g), salt (0.1g), olive oil (0.5g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564777960/rgb.png"}, {"dish_id": "dish_1563468269", "tier": "Low", "ground_truth_calories": 129.0, "total_mass_grams": 302.0, "total_fat_grams": 0.6, "total_carb_grams": 32.8, "total_protein_grams": 1.8, "num_ingredients": 3, "ingredient_summary": "berries (63.0g), watermelon (132.0g), pineapple (107.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563468269/rgb.png"}, {"dish_id": "dish_1562873028", "tier": "Low", "ground_truth_calories": 280.3, "total_mass_grams": 320.0, "total_fat_grams": 16.1, "total_carb_grams": 34.2, "total_protein_grams": 21.0, "num_ingredients": 18, "ingredient_summary": "jicama (3.2g), parsley (0.2g), nopales (2.4g), olive oil (4.1g), cherry tomatoes (54.0g), chive (0.2g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562873028/rgb.png"}, {"dish_id": "dish_1561580370", "tier": "Low", "ground_truth_calories": 126.8, "total_mass_grams": 124.0, "total_fat_grams": 5.0, "total_carb_grams": 16.2, "total_protein_grams": 5.4, "num_ingredients": 8, "ingredient_summary": "spinach (raw) (1.4g), wheat berry (23.0g), cheese (0.7g), cherry tomatoes (3.5g), pasta (35.3g), pesto (5.6g), +2 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561580370/rgb.png"}, {"dish_id": "dish_1561061658", "tier": "Low", "ground_truth_calories": 226.0, "total_mass_grams": 227.0, "total_fat_grams": 12.3, "total_carb_grams": 24.7, "total_protein_grams": 8.9, "num_ingredients": 17, "ingredient_summary": "croutons (0.3g), pepper (0.1g), parmesan cheese (4.2g), brussels sprouts (75.8g), mustard greens (34.3g), garlic (0.6g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561061658/rgb.png"}, {"dish_id": "dish_1559838402", "tier": "Low", "ground_truth_calories": 222.8, "total_mass_grams": 233.0, "total_fat_grams": 3.8, "total_carb_grams": 43.2, "total_protein_grams": 5.2, "num_ingredients": 4, "ingredient_summary": "bacon (8.0g), sweet potato (59.5g), yam (59.5g), berries (106.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559838402/rgb.png"}, {"dish_id": "dish_1561753238", "tier": "Low", "ground_truth_calories": 298.0, "total_mass_grams": 333.0, "total_fat_grams": 13.7, "total_carb_grams": 14.0, "total_protein_grams": 28.9, "num_ingredients": 14, "ingredient_summary": "olive oil (4.6g), arugula (4.1g), brown rice (20.7g), pepper (12.5g), squash (68.1g), kale (18.2g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561753238/rgb.png"}, {"dish_id": "dish_1565728415", "tier": "Low", "ground_truth_calories": 194.4, "total_mass_grams": 80.0, "total_fat_grams": 8.4, "total_carb_grams": 16.7, "total_protein_grams": 12.0, "num_ingredients": 10, "ingredient_summary": "cheese pizza (47.0g), shallots (1.7g), spinach (cooked) (0.8g), onions (0.8g), olive oil (0.2g), vinegar (0.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565728415/rgb.png"}, {"dish_id": "dish_1565383026", "tier": "Low", "ground_truth_calories": 171.6, "total_mass_grams": 287.0, "total_fat_grams": 14.0, "total_carb_grams": 12.2, "total_protein_grams": 3.7, "num_ingredients": 3, "ingredient_summary": "spinach (raw) (57.0g), cucumbers (101.0g), olives (129.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565383026/rgb.png"}, {"dish_id": "dish_1566849923", "tier": "Low", "ground_truth_calories": 128.0, "total_mass_grams": 240.0, "total_fat_grams": 7.5, "total_carb_grams": 16.2, "total_protein_grams": 2.1, "num_ingredients": 3, "ingredient_summary": "olives (67.0g), cherry tomatoes (87.0g), carrot (86.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566849923/rgb.png"}, {"dish_id": "dish_1560368972", "tier": "Low", "ground_truth_calories": 196.8, "total_mass_grams": 107.0, "total_fat_grams": 8.7, "total_carb_grams": 13.6, "total_protein_grams": 14.7, "num_ingredients": 15, "ingredient_summary": "pork (51.0g), cherry tomatoes (1.7g), garlic (0.7g), shallots (0.3g), white rice (41.0g), salt (0.1g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560368972/rgb.png"}, {"dish_id": "dish_1564675118", "tier": "Low", "ground_truth_calories": 177.2, "total_mass_grams": 238.0, "total_fat_grams": 4.1, "total_carb_grams": 34.1, "total_protein_grams": 2.0, "num_ingredients": 3, "ingredient_summary": "olive oil (3.9g), watermelon (77.0g), sweet potato (157.1g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564675118/rgb.png"}, {"dish_id": "dish_1567021831", "tier": "Low", "ground_truth_calories": 252.4, "total_mass_grams": 228.0, "total_fat_grams": 27.2, "total_carb_grams": 8.2, "total_protein_grams": 22.7, "num_ingredients": 18, "ingredient_summary": "cherry tomatoes (4.3g), pepper (0.1g), chive (0.3g), wheat berry (21.3g), kale (5.3g), mustard (0.6g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567021831/rgb.png"}, {"dish_id": "dish_1559846099", "tier": "Low", "ground_truth_calories": 206.9, "total_mass_grams": 114.0, "total_fat_grams": 14.6, "total_carb_grams": 15.7, "total_protein_grams": 7.7, "num_ingredients": 3, "ingredient_summary": "mandarin oranges (53.0g), almonds (29.0g), broccoli (32.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559846099/rgb.png"}, {"dish_id": "dish_1563900150", "tier": "Low", "ground_truth_calories": 202.3, "total_mass_grams": 154.0, "total_fat_grams": 12.7, "total_carb_grams": 10.3, "total_protein_grams": 11.0, "num_ingredients": 3, "ingredient_summary": "sweet potato (47.8g), olive oil (1.2g), scrambled eggs (105.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563900150/rgb.png"}, {"dish_id": "dish_1561662458", "tier": "Low", "ground_truth_calories": 124.8, "total_mass_grams": 119.0, "total_fat_grams": 4.9, "total_carb_grams": 21.1, "total_protein_grams": 3.2, "num_ingredients": 11, "ingredient_summary": "lettuce (6.5g), lime (0.3g), salt (0.2g), mixed greens (9.8g), onions (1.3g), cucumbers (3.3g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561662458/rgb.png"}, {"dish_id": "dish_1567106825", "tier": "Low", "ground_truth_calories": 220.2, "total_mass_grams": 104.0, "total_fat_grams": 16.9, "total_carb_grams": 3.4, "total_protein_grams": 13.5, "num_ingredients": 15, "ingredient_summary": "bok choy (1.4g), millet (5.7g), blue cheese (4.9g), onions (1.4g), mustard (0.2g), olive oil (6.8g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567106825/rgb.png"}, {"dish_id": "dish_1560973958", "tier": "Low", "ground_truth_calories": 145.3, "total_mass_grams": 121.0, "total_fat_grams": 6.2, "total_carb_grams": 20.7, "total_protein_grams": 4.3, "num_ingredients": 10, "ingredient_summary": "wheat berry (77.0g), olive oil (3.7g), mustard (0.2g), pecans (2.7g), lime (0.7g), salt (0.1g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560973958/rgb.png"}, {"dish_id": "dish_1558546663", "tier": "Low", "ground_truth_calories": 330.7, "total_mass_grams": 197.0, "total_fat_grams": 22.1, "total_carb_grams": 30.3, "total_protein_grams": 10.6, "num_ingredients": 3, "ingredient_summary": "spinach (raw) (11.0g), almonds (44.0g), apple (142.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558546663/rgb.png"}, {"dish_id": "dish_1563468299", "tier": "Low", "ground_truth_calories": 210.4, "total_mass_grams": 385.0, "total_fat_grams": 4.1, "total_carb_grams": 35.6, "total_protein_grams": 10.9, "num_ingredients": 4, "ingredient_summary": "cottage cheese (83.0g), pineapple (107.0g), watermelon (132.0g), berries (63.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563468299/rgb.png"}, {"dish_id": "dish_1563206582", "tier": "Low", "ground_truth_calories": 229.2, "total_mass_grams": 320.0, "total_fat_grams": 4.4, "total_carb_grams": 17.0, "total_protein_grams": 26.0, "num_ingredients": 4, "ingredient_summary": "yam (32.1g), egg whites (252.0g), olive oil (3.8g), sweet potato (32.1g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563206582/rgb.png"}, {"dish_id": "dish_1561665024", "tier": "Low", "ground_truth_calories": 100.6, "total_mass_grams": 121.0, "total_fat_grams": 7.0, "total_carb_grams": 8.9, "total_protein_grams": 2.1, "num_ingredients": 10, "ingredient_summary": "lemon juice (1.8g), onions (3.5g), olive oil (6.7g), mixed greens (43.9g), parsley (0.4g), salt (0.2g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561665024/rgb.png"}, {"dish_id": "dish_1563475921", "tier": "Low", "ground_truth_calories": 244.3, "total_mass_grams": 348.0, "total_fat_grams": 6.7, "total_carb_grams": 27.2, "total_protein_grams": 19.8, "num_ingredients": 17, "ingredient_summary": "shallots (2.1g), garlic (2.1g), lemon juice (0.7g), onions (1.4g), spinach (raw) (3.5g), jalapenos (0.7g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563475921/rgb.png"}, {"dish_id": "dish_1561739954", "tier": "Low", "ground_truth_calories": 276.1, "total_mass_grams": 225.0, "total_fat_grams": 16.1, "total_carb_grams": 16.6, "total_protein_grams": 15.4, "num_ingredients": 2, "ingredient_summary": "sweet potato (79.0g), scrambled eggs (146.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561739954/rgb.png"}, {"dish_id": "dish_1563813308", "tier": "Low", "ground_truth_calories": 225.0, "total_mass_grams": 76.0, "total_fat_grams": 16.4, "total_carb_grams": 3.9, "total_protein_grams": 15.3, "num_ingredients": 5, "ingredient_summary": "salt (0.2g), pepper (0.0g), brussels sprouts (37.4g), olive oil (0.4g), bacon (38.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563813308/rgb.png"}, {"dish_id": "dish_1564689108", "tier": "Low", "ground_truth_calories": 203.7, "total_mass_grams": 159.0, "total_fat_grams": 15.9, "total_carb_grams": 13.2, "total_protein_grams": 4.9, "num_ingredients": 15, "ingredient_summary": "chickpeas (20.9g), salt (0.4g), garlic (0.5g), blueberries (7.1g), pepper (6.3g), pears (10.4g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564689108/rgb.png"}, {"dish_id": "dish_1562602860", "tier": "Low", "ground_truth_calories": 165.2, "total_mass_grams": 258.0, "total_fat_grams": 5.6, "total_carb_grams": 20.8, "total_protein_grams": 10.2, "num_ingredients": 5, "ingredient_summary": "strawberries (68.0g), milk (55.3g), chia seeds (13.0g), greek yogurt (51.7g), watermelon (70.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562602860/rgb.png"}, {"dish_id": "dish_1565898158", "tier": "Low", "ground_truth_calories": 240.4, "total_mass_grams": 194.0, "total_fat_grams": 10.3, "total_carb_grams": 10.2, "total_protein_grams": 25.3, "num_ingredients": 15, "ingredient_summary": "cilantro (1.0g), ginger (0.5g), spinach (raw) (4.9g), jalapenos (1.0g), lemon juice (1.0g), lime (0.1g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565898158/rgb.png"}, {"dish_id": "dish_1565022062", "tier": "Low", "ground_truth_calories": 234.6, "total_mass_grams": 366.0, "total_fat_grams": 2.8, "total_carb_grams": 30.9, "total_protein_grams": 19.3, "num_ingredients": 4, "ingredient_summary": "olive oil (2.2g), egg whites (177.0g), berries (99.0g), sweet potato (87.8g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565022062/rgb.png"}, {"dish_id": "dish_1561663855", "tier": "Low", "ground_truth_calories": 179.1, "total_mass_grams": 139.0, "total_fat_grams": 7.4, "total_carb_grams": 14.4, "total_protein_grams": 15.3, "num_ingredients": 18, "ingredient_summary": "parsley (0.3g), lemon juice (1.2g), chive (0.1g), garlic (0.2g), millet (4.0g), apple (5.0g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561663855/rgb.png"}, {"dish_id": "dish_1566488928", "tier": "Low", "ground_truth_calories": 103.0, "total_mass_grams": 147.0, "total_fat_grams": 2.2, "total_carb_grams": 20.0, "total_protein_grams": 1.3, "num_ingredients": 3, "ingredient_summary": "cantaloupe (61.0g), sweet potato (83.9g), olive oil (2.1g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566488928/rgb.png"}, {"dish_id": "dish_1563392128", "tier": "Low", "ground_truth_calories": 169.9, "total_mass_grams": 162.0, "total_fat_grams": 18.8, "total_carb_grams": 6.5, "total_protein_grams": 21.1, "num_ingredients": 12, "ingredient_summary": "garlic (0.5g), cilantro (1.1g), lemon juice (2.1g), onions (4.3g), olive oil (4.3g), salt (0.3g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563392128/rgb.png"}, {"dish_id": "dish_1560368464", "tier": "Low", "ground_truth_calories": 349.4, "total_mass_grams": 275.0, "total_fat_grams": 18.1, "total_carb_grams": 16.8, "total_protein_grams": 29.1, "num_ingredients": 15, "ingredient_summary": "olive oil (2.8g), brown rice (14.2g), vinegar (0.7g), cherry tomatoes (3.6g), honeydew melons (137.0g), shallots (0.7g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560368464/rgb.png"}, {"dish_id": "dish_1560368990", "tier": "Low", "ground_truth_calories": 211.9, "total_mass_grams": 119.0, "total_fat_grams": 10.0, "total_carb_grams": 14.4, "total_protein_grams": 15.0, "num_ingredients": 21, "ingredient_summary": "shallots (0.3g), salt (0.1g), pumpkin seeds (0.6g), parsley (0.2g), olive oil (2.5g), brown rice (6.8g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560368990/rgb.png"}, {"dish_id": "dish_1562958563", "tier": "Low", "ground_truth_calories": 130.8, "total_mass_grams": 68.0, "total_fat_grams": 9.8, "total_carb_grams": 2.7, "total_protein_grams": 8.2, "num_ingredients": 2, "ingredient_summary": "olives (21.0g), chicken salad (47.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562958563/rgb.png"}, {"dish_id": "dish_1561579617", "tier": "Low", "ground_truth_calories": 130.2, "total_mass_grams": 304.0, "total_fat_grams": 4.6, "total_carb_grams": 21.1, "total_protein_grams": 4.4, "num_ingredients": 5, "ingredient_summary": "salt (1.4g), eggplant (138.2g), olive oil (1.4g), cauliflower (126.0g), roasted potatoes (37.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561579617/rgb.png"}, {"dish_id": "dish_1566333167", "tier": "Low", "ground_truth_calories": 344.5, "total_mass_grams": 140.0, "total_fat_grams": 14.5, "total_carb_grams": 31.6, "total_protein_grams": 20.3, "num_ingredients": 10, "ingredient_summary": "vinegar (0.1g), beef (38.2g), bok choy (1.3g), spinach (cooked) (1.3g), shallots (2.5g), millet (5.1g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566333167/rgb.png"}, {"dish_id": "dish_1565033139", "tier": "Low", "ground_truth_calories": 193.6, "total_mass_grams": 274.0, "total_fat_grams": 14.2, "total_carb_grams": 17.4, "total_protein_grams": 4.4, "num_ingredients": 3, "ingredient_summary": "mixed greens (55.0g), broccoli (92.0g), olives (127.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565033139/rgb.png"}, {"dish_id": "dish_1563464480", "tier": "Low", "ground_truth_calories": 229.5, "total_mass_grams": 213.0, "total_fat_grams": 10.9, "total_carb_grams": 25.8, "total_protein_grams": 8.3, "num_ingredients": 4, "ingredient_summary": "bacon (17.0g), watermelon (60.0g), roasted potatoes (50.0g), berries (86.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563464480/rgb.png"}, {"dish_id": "dish_1567020054", "tier": "Low", "ground_truth_calories": 122.0, "total_mass_grams": 145.0, "total_fat_grams": 11.5, "total_carb_grams": 7.9, "total_protein_grams": 14.2, "num_ingredients": 14, "ingredient_summary": "shallots (1.3g), garlic (0.0g), parsley (0.6g), salt (0.4g), kale (3.2g), brussels sprouts (54.1g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567020054/rgb.png"}, {"dish_id": "dish_1568649387", "tier": "Low", "ground_truth_calories": 163.9, "total_mass_grams": 192.0, "total_fat_grams": 5.3, "total_carb_grams": 28.5, "total_protein_grams": 2.5, "num_ingredients": 5, "ingredient_summary": "olive oil (4.8g), sweet potato (29.7g), broccoli (49.8g), berries (78.0g), yam (29.7g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568649387/rgb.png"}, {"dish_id": "dish_1566499633", "tier": "Low", "ground_truth_calories": 161.4, "total_mass_grams": 139.0, "total_fat_grams": 14.2, "total_carb_grams": 8.4, "total_protein_grams": 3.0, "num_ingredients": 10, "ingredient_summary": "onions (7.4g), olive oil (9.8g), vinegar (4.9g), garlic (0.6g), salt (0.3g), sun dried tomatoes (4.9g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566499633/rgb.png"}, {"dish_id": "dish_1558549214", "tier": "Low", "ground_truth_calories": 239.2, "total_mass_grams": 361.0, "total_fat_grams": 0.7, "total_carb_grams": 53.9, "total_protein_grams": 5.7, "num_ingredients": 5, "ingredient_summary": "white rice (75.0g), potatoes (95.0g), spinach (raw) (10.0g), cherry tomatoes (115.0g), grapes (66.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558549214/rgb.png"}, {"dish_id": "dish_1562008813", "tier": "Low", "ground_truth_calories": 214.6, "total_mass_grams": 225.0, "total_fat_grams": 15.6, "total_carb_grams": 15.6, "total_protein_grams": 6.0, "num_ingredients": 19, "ingredient_summary": "pears (9.4g), quinoa (25.6g), shallots (0.1g), olive oil (11.2g), lemon juice (0.1g), mustard (1.2g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562008813/rgb.png"}, {"dish_id": "dish_1568061202", "tier": "Low", "ground_truth_calories": 256.6, "total_mass_grams": 334.0, "total_fat_grams": 7.7, "total_carb_grams": 39.2, "total_protein_grams": 9.4, "num_ingredients": 7, "ingredient_summary": "salt (0.8g), pepper (0.1g), brussels sprouts (46.2g), olive oil (1.9g), squash (139.1g), pineapple (91.0g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568061202/rgb.png"}, {"dish_id": "dish_1563900172", "tier": "Low", "ground_truth_calories": 236.6, "total_mass_grams": 255.0, "total_fat_grams": 12.9, "total_carb_grams": 18.4, "total_protein_grams": 11.8, "num_ingredients": 4, "ingredient_summary": "scrambled eggs (105.0g), cantaloupe (101.0g), olive oil (1.2g), sweet potato (47.8g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563900172/rgb.png"}, {"dish_id": "dish_1562785132", "tier": "Low", "ground_truth_calories": 211.5, "total_mass_grams": 192.0, "total_fat_grams": 24.6, "total_carb_grams": 10.4, "total_protein_grams": 29.0, "num_ingredients": 7, "ingredient_summary": "fish (150.3g), country rice (30.1g), onions (6.0g), olive oil (3.0g), garlic (0.8g), salt (0.4g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562785132/rgb.png"}, {"dish_id": "dish_1563913675", "tier": "Low", "ground_truth_calories": 282.7, "total_mass_grams": 228.0, "total_fat_grams": 12.5, "total_carb_grams": 31.8, "total_protein_grams": 12.2, "num_ingredients": 11, "ingredient_summary": "spinach (raw) (13.3g), vinegar (1.8g), chard (8.8g), cherry tomatoes (4.4g), cheese pizza (68.0g), figs (4.4g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563913675/rgb.png"}, {"dish_id": "dish_1559244305", "tier": "Low", "ground_truth_calories": 113.1, "total_mass_grams": 64.0, "total_fat_grams": 7.0, "total_carb_grams": 9.8, "total_protein_grams": 3.5, "num_ingredients": 8, "ingredient_summary": "arugula (8.0g), pepper (3.2g), vinegar (3.2g), chickpeas (32.1g), salt (0.2g), parsley (3.2g), +2 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559244305/rgb.png"}, {"dish_id": "dish_1563550614", "tier": "Low", "ground_truth_calories": 132.8, "total_mass_grams": 205.0, "total_fat_grams": 1.8, "total_carb_grams": 11.6, "total_protein_grams": 15.1, "num_ingredients": 3, "ingredient_summary": "sweet potato (58.5g), olive oil (1.5g), egg whites (145.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563550614/rgb.png"}, {"dish_id": "dish_1558724031", "tier": "Medium-Low", "ground_truth_calories": 358.7, "total_mass_grams": 303.0, "total_fat_grams": 25.8, "total_carb_grams": 25.8, "total_protein_grams": 14.1, "num_ingredients": 4, "ingredient_summary": "cherry tomatoes (107.0g), cantaloupe (93.0g), almonds (51.0g), cauliflower (52.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558724031/rgb.png"}, {"dish_id": "dish_1562789268", "tier": "Medium-Low", "ground_truth_calories": 569.5, "total_mass_grams": 350.0, "total_fat_grams": 22.7, "total_carb_grams": 64.8, "total_protein_grams": 32.6, "num_ingredients": 16, "ingredient_summary": "pork (76.4g), corn on the cob (196.0g), brown rice (10.2g), lemon juice (0.5g), parsley (0.3g), basil (0.3g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562789268/rgb.png"}, {"dish_id": "dish_1561662054", "tier": "Medium-Low", "ground_truth_calories": 419.4, "total_mass_grams": 292.0, "total_fat_grams": 23.8, "total_carb_grams": 26.4, "total_protein_grams": 25.9, "num_ingredients": 17, "ingredient_summary": "pepper (0.3g), white rice (12.0g), mixed greens (75.3g), garlic (3.0g), soy sauce (4.8g), millet (12.0g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561662054/rgb.png"}, {"dish_id": "dish_1564432238", "tier": "Medium-Low", "ground_truth_calories": 455.4, "total_mass_grams": 315.0, "total_fat_grams": 23.2, "total_carb_grams": 34.9, "total_protein_grams": 28.1, "num_ingredients": 22, "ingredient_summary": "cherry tomatoes (3.6g), cauliflower (60.0g), lime (0.5g), mayonnaise (1.8g), jalapenos (0.1g), pepperoni pizza (63.0g), +16 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564432238/rgb.png"}, {"dish_id": "dish_1563468327", "tier": "Medium-Low", "ground_truth_calories": 351.9, "total_mass_grams": 483.0, "total_fat_grams": 9.8, "total_carb_grams": 57.5, "total_protein_grams": 12.1, "num_ingredients": 7, "ingredient_summary": "yam (46.2g), sweet potato (46.2g), cottage cheese (83.0g), pineapple (107.0g), watermelon (132.0g), olive oil (5.5g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563468327/rgb.png"}, {"dish_id": "dish_1566417516", "tier": "Medium-Low", "ground_truth_calories": 385.4, "total_mass_grams": 425.0, "total_fat_grams": 36.5, "total_carb_grams": 23.5, "total_protein_grams": 33.7, "num_ingredients": 17, "ingredient_summary": "wheat berry (31.6g), eggplant (109.8g), cherry tomatoes (8.9g), mixed greens (32.5g), celery root (2.6g), salt (1.6g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566417516/rgb.png"}, {"dish_id": "dish_1562008979", "tier": "Medium-Low", "ground_truth_calories": 382.9, "total_mass_grams": 290.0, "total_fat_grams": 22.2, "total_carb_grams": 10.2, "total_protein_grams": 35.3, "num_ingredients": 13, "ingredient_summary": "jalapenos (0.2g), lemon juice (1.5g), pork (115.8g), wheat berry (15.4g), cabbage (15.4g), pepper (0.4g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562008979/rgb.png"}, {"dish_id": "dish_1566920365", "tier": "Medium-Low", "ground_truth_calories": 490.4, "total_mass_grams": 871.0, "total_fat_grams": 5.3, "total_carb_grams": 41.4, "total_protein_grams": 60.7, "num_ingredients": 4, "ingredient_summary": "salsa (141.0g), sweet potato (156.1g), olive oil (3.9g), egg whites (570.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566920365/rgb.png"}, {"dish_id": "dish_1563984242", "tier": "Medium-Low", "ground_truth_calories": 402.1, "total_mass_grams": 753.0, "total_fat_grams": 3.3, "total_carb_grams": 36.8, "total_protein_grams": 49.4, "num_ingredients": 5, "ingredient_summary": "sweet potato (81.0g), egg whites (472.0g), pineapple (62.0g), olive oil (2.0g), cantaloupe (136.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563984242/rgb.png"}, {"dish_id": "dish_1562691032", "tier": "Medium-Low", "ground_truth_calories": 419.7, "total_mass_grams": 415.0, "total_fat_grams": 22.0, "total_carb_grams": 39.1, "total_protein_grams": 17.9, "num_ingredients": 6, "ingredient_summary": "berries (85.0g), olive oil (0.9g), pineapple (80.0g), scrambled eggs (147.0g), roasted potatoes (66.0g), broccoli (36.1g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562691032/rgb.png"}, {"dish_id": "dish_1563998323", "tier": "Medium-Low", "ground_truth_calories": 634.5, "total_mass_grams": 574.0, "total_fat_grams": 32.5, "total_carb_grams": 35.3, "total_protein_grams": 53.0, "num_ingredients": 26, "ingredient_summary": "roasted potatoes (79.0g), shallots (1.3g), olive oil (4.4g), tatsoi (4.6g), butter (0.7g), jalapenos (0.5g), +20 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563998323/rgb.png"}, {"dish_id": "dish_1561492204", "tier": "Medium-Low", "ground_truth_calories": 488.6, "total_mass_grams": 334.0, "total_fat_grams": 15.7, "total_carb_grams": 49.1, "total_protein_grams": 37.2, "num_ingredients": 16, "ingredient_summary": "olive oil (9.0g), blueberries (4.4g), onions (4.6g), tomatoes (11.6g), salt (2.5g), fried rice (128.0g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561492204/rgb.png"}, {"dish_id": "dish_1567613057", "tier": "Medium-Low", "ground_truth_calories": 406.7, "total_mass_grams": 749.0, "total_fat_grams": 5.0, "total_carb_grams": 45.9, "total_protein_grams": 40.7, "num_ingredients": 5, "ingredient_summary": "olive oil (3.8g), egg whites (369.0g), watermelon (121.0g), sweet potato (151.2g), salsa (104.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567613057/rgb.png"}, {"dish_id": "dish_1558724959", "tier": "Medium-Low", "ground_truth_calories": 581.4, "total_mass_grams": 426.0, "total_fat_grams": 36.5, "total_carb_grams": 57.0, "total_protein_grams": 21.3, "num_ingredients": 4, "ingredient_summary": "brussels sprouts (118.0g), carrot (75.0g), almonds (72.0g), apple (161.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558724959/rgb.png"}, {"dish_id": "dish_1566938026", "tier": "Medium-Low", "ground_truth_calories": 450.6, "total_mass_grams": 219.0, "total_fat_grams": 17.5, "total_carb_grams": 37.5, "total_protein_grams": 33.5, "num_ingredients": 13, "ingredient_summary": "spinach (cooked) (2.2g), wine (0.4g), cheese pizza (104.0g), salt (0.1g), lemon juice (0.1g), squash (4.4g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566938026/rgb.png"}, {"dish_id": "dish_1564000490", "tier": "Medium-Low", "ground_truth_calories": 532.7, "total_mass_grams": 325.0, "total_fat_grams": 19.2, "total_carb_grams": 49.6, "total_protein_grams": 39.9, "num_ingredients": 19, "ingredient_summary": "pepper (0.3g), arugula (1.7g), cream (2.8g), caesar salad (108.0g), butter (0.7g), shallots (0.3g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564000490/rgb.png"}, {"dish_id": "dish_1559751801", "tier": "Medium-Low", "ground_truth_calories": 451.6, "total_mass_grams": 343.0, "total_fat_grams": 22.0, "total_carb_grams": 32.8, "total_protein_grams": 29.2, "num_ingredients": 4, "ingredient_summary": "berries (66.0g), yam (66.0g), eggs (122.0g), chicken apple sausage (89.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559751801/rgb.png"}, {"dish_id": "dish_1564000459", "tier": "Medium-Low", "ground_truth_calories": 485.2, "total_mass_grams": 217.0, "total_fat_grams": 16.9, "total_carb_grams": 45.0, "total_protein_grams": 36.5, "num_ingredients": 18, "ingredient_summary": "butter (0.7g), wheat berry (6.9g), thyme (0.2g), olive oil (0.7g), rosemary (0.2g), lemon juice (0.2g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564000459/rgb.png"}, {"dish_id": "dish_1567628157", "tier": "Medium-Low", "ground_truth_calories": 373.2, "total_mass_grams": 355.0, "total_fat_grams": 36.0, "total_carb_grams": 25.9, "total_protein_grams": 37.5, "num_ingredients": 17, "ingredient_summary": "green onions (1.7g), fish (174.6g), apple (8.5g), arugula (17.0g), onions (7.0g), chard (29.5g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567628157/rgb.png"}, {"dish_id": "dish_1566502505", "tier": "Medium-Low", "ground_truth_calories": 454.5, "total_mass_grams": 275.0, "total_fat_grams": 22.5, "total_carb_grams": 33.1, "total_protein_grams": 31.9, "num_ingredients": 16, "ingredient_summary": "sun dried tomatoes (3.1g), bok choy (14.9g), pork (104.2g), corn on the cob (55.0g), cucumbers (7.7g), onions (4.6g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566502505/rgb.png"}, {"dish_id": "dish_1562959732", "tier": "Medium-Low", "ground_truth_calories": 374.7, "total_mass_grams": 232.0, "total_fat_grams": 14.7, "total_carb_grams": 31.1, "total_protein_grams": 28.9, "num_ingredients": 9, "ingredient_summary": "salt (0.7g), soy sauce (5.8g), bok choy (14.6g), pork (102.2g), parsnips (79.0g), pepper (0.4g), +3 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562959732/rgb.png"}, {"dish_id": "dish_1566502537", "tier": "Medium-Low", "ground_truth_calories": 517.6, "total_mass_grams": 315.0, "total_fat_grams": 24.3, "total_carb_grams": 34.3, "total_protein_grams": 41.8, "num_ingredients": 25, "ingredient_summary": "corn on the cob (55.0g), vinegar (3.1g), parsley (0.0g), sun dried tomatoes (3.1g), rosemary (0.0g), carrot (3.2g), +19 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566502537/rgb.png"}, {"dish_id": "dish_1564778030", "tier": "Medium-Low", "ground_truth_calories": 504.4, "total_mass_grams": 283.0, "total_fat_grams": 31.9, "total_carb_grams": 23.5, "total_protein_grams": 30.8, "num_ingredients": 17, "ingredient_summary": "millet (11.0g), spinach (cooked) (2.7g), vinegar (4.4g), onions (2.7g), cheese pizza (39.0g), broccoli (72.2g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564778030/rgb.png"}, {"dish_id": "dish_1566838407", "tier": "Medium-Low", "ground_truth_calories": 358.9, "total_mass_grams": 558.0, "total_fat_grams": 9.8, "total_carb_grams": 62.4, "total_protein_grams": 9.7, "num_ingredients": 6, "ingredient_summary": "pineapple (145.0g), scrambled eggs (65.0g), sweet potato (74.1g), berries (133.0g), olive oil (1.9g), watermelon (139.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566838407/rgb.png"}, {"dish_id": "dish_1564429894", "tier": "Medium-Low", "ground_truth_calories": 498.8, "total_mass_grams": 566.0, "total_fat_grams": 11.8, "total_carb_grams": 73.5, "total_protein_grams": 25.3, "num_ingredients": 19, "ingredient_summary": "quinoa (184.0g), salt (2.3g), squash (51.3g), mayonnaise (0.9g), arugula (18.4g), parsley (3.9g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564429894/rgb.png"}, {"dish_id": "dish_1560360055", "tier": "Medium-Low", "ground_truth_calories": 362.7, "total_mass_grams": 180.0, "total_fat_grams": 27.8, "total_carb_grams": 4.2, "total_protein_grams": 22.6, "num_ingredients": 2, "ingredient_summary": "sausage (49.0g), scrambled eggs (131.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560360055/rgb.png"}, {"dish_id": "dish_1560453380", "tier": "Medium-Low", "ground_truth_calories": 411.4, "total_mass_grams": 480.0, "total_fat_grams": 15.2, "total_carb_grams": 25.5, "total_protein_grams": 41.9, "num_ingredients": 20, "ingredient_summary": "zucchini (2.4g), brown rice (18.9g), arugula (3.8g), green beans (72.0g), wheat berry (4.7g), rosemary (0.1g), +14 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560453380/rgb.png"}, {"dish_id": "dish_1566849327", "tier": "Medium-Low", "ground_truth_calories": 551.7, "total_mass_grams": 362.0, "total_fat_grams": 31.3, "total_carb_grams": 14.9, "total_protein_grams": 55.1, "num_ingredients": 5, "ingredient_summary": "parmesan cheese (36.0g), mixed greens (70.0g), avocado (47.0g), olives (86.0g), chicken (123.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566849327/rgb.png"}, {"dish_id": "dish_1563898786", "tier": "Medium-Low", "ground_truth_calories": 420.9, "total_mass_grams": 344.0, "total_fat_grams": 31.5, "total_carb_grams": 11.4, "total_protein_grams": 23.6, "num_ingredients": 10, "ingredient_summary": "mustard (0.5g), olive oil (8.0g), bell peppers (9.9g), broccoli (93.0g), radishes (8.0g), salt (0.2g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563898786/rgb.png"}, {"dish_id": "dish_1565119464", "tier": "Medium-Low", "ground_truth_calories": 504.1, "total_mass_grams": 223.0, "total_fat_grams": 23.1, "total_carb_grams": 44.4, "total_protein_grams": 28.0, "num_ingredients": 14, "ingredient_summary": "millet (6.8g), apple (1.7g), cheese pizza (122.0g), salt (0.1g), olive oil (3.8g), onions (5.1g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565119464/rgb.png"}, {"dish_id": "dish_1559841163", "tier": "Medium-Low", "ground_truth_calories": 357.0, "total_mass_grams": 335.0, "total_fat_grams": 18.2, "total_carb_grams": 27.5, "total_protein_grams": 21.2, "num_ingredients": 6, "ingredient_summary": "blackberries (53.0g), scrambled eggs (119.0g), yam (33.5g), sweet potato (33.5g), chicken apple sausage (46.0g), broccoli (50.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559841163/rgb.png"}, {"dish_id": "dish_1563568338", "tier": "Medium-Low", "ground_truth_calories": 631.7, "total_mass_grams": 371.0, "total_fat_grams": 29.6, "total_carb_grams": 57.6, "total_protein_grams": 38.7, "num_ingredients": 17, "ingredient_summary": "salt (0.4g), garlic (0.2g), olive oil (2.1g), kale (3.5g), onions (3.5g), wheat berry (23.3g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563568338/rgb.png"}, {"dish_id": "dish_1562691674", "tier": "Medium-Low", "ground_truth_calories": 377.0, "total_mass_grams": 435.0, "total_fat_grams": 11.3, "total_carb_grams": 42.6, "total_protein_grams": 26.5, "num_ingredients": 9, "ingredient_summary": "sweet potato (56.5g), olive oil (2.2g), broccoli (88.8g), egg whites (96.0g), yam (56.5g), greek yogurt (38.8g), +3 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562691674/rgb.png"}, {"dish_id": "dish_1563216440", "tier": "Medium-Low", "ground_truth_calories": 403.8, "total_mass_grams": 352.0, "total_fat_grams": 6.5, "total_carb_grams": 39.9, "total_protein_grams": 44.7, "num_ingredients": 11, "ingredient_summary": "wheat berry (12.1g), olive oil (0.5g), spinach (raw) (3.0g), cherry tomatoes (6.0g), chicken breast (120.7g), white rice (106.0g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563216440/rgb.png"}, {"dish_id": "dish_1563395480", "tier": "Medium-Low", "ground_truth_calories": 370.5, "total_mass_grams": 329.0, "total_fat_grams": 14.1, "total_carb_grams": 31.8, "total_protein_grams": 29.2, "num_ingredients": 27, "ingredient_summary": "corn (3.0g), cheese (1.8g), garlic (0.7g), chili (4.6g), black beans (6.1g), spinach (raw) (15.1g), +21 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563395480/rgb.png"}, {"dish_id": "dish_1559235815", "tier": "Medium-Low", "ground_truth_calories": 352.4, "total_mass_grams": 268.0, "total_fat_grams": 16.3, "total_carb_grams": 31.3, "total_protein_grams": 19.6, "num_ingredients": 4, "ingredient_summary": "cream cheese (10.8g), bagels (43.2g), strawberries (103.0g), eggs (111.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559235815/rgb.png"}, {"dish_id": "dish_1565640272", "tier": "Medium-Low", "ground_truth_calories": 377.8, "total_mass_grams": 405.0, "total_fat_grams": 13.9, "total_carb_grams": 49.7, "total_protein_grams": 16.6, "num_ingredients": 3, "ingredient_summary": "bell peppers (54.0g), pepperoni pizza (111.0g), cherry tomatoes (240.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565640272/rgb.png"}, {"dish_id": "dish_1562960520", "tier": "Medium-Low", "ground_truth_calories": 422.4, "total_mass_grams": 303.0, "total_fat_grams": 19.7, "total_carb_grams": 31.7, "total_protein_grams": 29.5, "num_ingredients": 10, "ingredient_summary": "pork (91.6g), pepper (0.3g), white rice (13.1g), garlic (3.3g), caesar salad (97.0g), roasted potatoes (69.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562960520/rgb.png"}, {"dish_id": "dish_1563389199", "tier": "Medium-Low", "ground_truth_calories": 390.3, "total_mass_grams": 284.0, "total_fat_grams": 8.3, "total_carb_grams": 70.2, "total_protein_grams": 16.5, "num_ingredients": 14, "ingredient_summary": "broccoli (57.0g), garlic (0.1g), pepper (0.0g), arugula (19.5g), vinegar (0.7g), chard (3.5g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563389199/rgb.png"}, {"dish_id": "dish_1563910171", "tier": "Medium-Low", "ground_truth_calories": 571.2, "total_mass_grams": 423.0, "total_fat_grams": 27.3, "total_carb_grams": 24.9, "total_protein_grams": 53.6, "num_ingredients": 21, "ingredient_summary": "ginger (1.0g), chard (18.1g), salmon (204.5g), cilantro (2.0g), shallots (6.1g), lime (0.3g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563910171/rgb.png"}, {"dish_id": "dish_1563551194", "tier": "Medium-Low", "ground_truth_calories": 471.6, "total_mass_grams": 400.0, "total_fat_grams": 14.0, "total_carb_grams": 57.7, "total_protein_grams": 27.8, "num_ingredients": 8, "ingredient_summary": "olive oil (2.4g), sweet potato (94.6g), egg whites (101.0g), bagels (57.6g), greek yogurt (56.0g), chia seeds (14.0g), +2 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563551194/rgb.png"}, {"dish_id": "dish_1565370004", "tier": "Medium-Low", "ground_truth_calories": 570.9, "total_mass_grams": 370.0, "total_fat_grams": 41.7, "total_carb_grams": 11.2, "total_protein_grams": 36.9, "num_ingredients": 4, "ingredient_summary": "bacon (31.0g), olive oil (2.6g), scrambled eggs (236.0g), green beans (100.4g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565370004/rgb.png"}, {"dish_id": "dish_1562602657", "tier": "Medium-Low", "ground_truth_calories": 610.4, "total_mass_grams": 236.0, "total_fat_grams": 44.1, "total_carb_grams": 31.0, "total_protein_grams": 24.3, "num_ingredients": 6, "ingredient_summary": "pepper (0.1g), salt (0.3g), hash browns (65.0g), brussels sprouts (64.0g), sausage (106.0g), olive oil (0.6g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562602657/rgb.png"}, {"dish_id": "dish_1559668314", "tier": "Medium-Low", "ground_truth_calories": 551.5, "total_mass_grams": 300.0, "total_fat_grams": 38.2, "total_carb_grams": 32.8, "total_protein_grams": 21.0, "num_ingredients": 3, "ingredient_summary": "scrambled eggs (150.0g), garden salad (61.0g), hash browns (89.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559668314/rgb.png"}, {"dish_id": "dish_1563306392", "tier": "Medium-Low", "ground_truth_calories": 477.5, "total_mass_grams": 288.0, "total_fat_grams": 14.0, "total_carb_grams": 5.7, "total_protein_grams": 77.8, "num_ingredients": 10, "ingredient_summary": "chicken (246.4g), spinach (raw) (6.2g), wheat berry (24.6g), thyme (0.6g), olive oil (4.9g), lemon juice (1.2g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563306392/rgb.png"}, {"dish_id": "dish_1565383097", "tier": "Medium-Low", "ground_truth_calories": 479.2, "total_mass_grams": 430.0, "total_fat_grams": 25.9, "total_carb_grams": 13.3, "total_protein_grams": 50.0, "num_ingredients": 5, "ingredient_summary": "spinach (raw) (57.0g), cucumbers (101.0g), olives (129.0g), chicken (116.0g), parmesan cheese (27.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565383097/rgb.png"}, {"dish_id": "dish_1568401261", "tier": "Medium-Low", "ground_truth_calories": 530.4, "total_mass_grams": 443.0, "total_fat_grams": 26.4, "total_carb_grams": 34.6, "total_protein_grams": 39.1, "num_ingredients": 6, "ingredient_summary": "salt (0.4g), squash (113.4g), carrot (75.0g), avocado (25.0g), tuna salad (228.0g), olive oil (1.1g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568401261/rgb.png"}, {"dish_id": "dish_1562790824", "tier": "Medium-Low", "ground_truth_calories": 437.1, "total_mass_grams": 368.0, "total_fat_grams": 49.8, "total_carb_grams": 19.3, "total_protein_grams": 49.9, "num_ingredients": 12, "ingredient_summary": "salt (0.9g), bell peppers (9.7g), garlic (1.8g), mustard (0.5g), cucumbers (9.7g), olive oil (12.9g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562790824/rgb.png"}, {"dish_id": "dish_1562959129", "tier": "Medium-Low", "ground_truth_calories": 412.9, "total_mass_grams": 293.0, "total_fat_grams": 22.2, "total_carb_grams": 47.4, "total_protein_grams": 24.7, "num_ingredients": 16, "ingredient_summary": "fish (68.9g), garlic (0.3g), salt (0.2g), pizza (79.0g), onions (2.8g), pepper (2.8g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562959129/rgb.png"}, {"dish_id": "dish_1560203424", "tier": "Medium-Low", "ground_truth_calories": 451.8, "total_mass_grams": 138.0, "total_fat_grams": 22.3, "total_carb_grams": 50.2, "total_protein_grams": 14.6, "num_ingredients": 2, "ingredient_summary": "pepperoni pizza (104.0g), brownies (34.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560203424/rgb.png"}, {"dish_id": "dish_1567542876", "tier": "Medium-Low", "ground_truth_calories": 495.8, "total_mass_grams": 260.0, "total_fat_grams": 21.0, "total_carb_grams": 45.7, "total_protein_grams": 29.5, "num_ingredients": 18, "ingredient_summary": "white rice (25.7g), soy sauce (0.4g), onions (7.2g), orange juice (0.4g), kale (6.4g), lemon juice (0.9g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567542876/rgb.png"}, {"dish_id": "dish_1565638323", "tier": "Medium-Low", "ground_truth_calories": 467.8, "total_mass_grams": 274.0, "total_fat_grams": 20.7, "total_carb_grams": 29.2, "total_protein_grams": 38.3, "num_ingredients": 11, "ingredient_summary": "salt (0.2g), garlic (0.9g), parsley (0.4g), onions (6.3g), zucchini (8.9g), rosemary (0.9g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565638323/rgb.png"}, {"dish_id": "dish_1565898659", "tier": "Medium-Low", "ground_truth_calories": 576.7, "total_mass_grams": 537.0, "total_fat_grams": 11.4, "total_carb_grams": 16.1, "total_protein_grams": 97.2, "num_ingredients": 18, "ingredient_summary": "cherry tomatoes (62.0g), tuna (280.0g), olives (40.0g), salmon (58.0g), spinach (raw) (2.9g), onions (1.2g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565898659/rgb.png"}, {"dish_id": "dish_1562698276", "tier": "Medium-Low", "ground_truth_calories": 434.6, "total_mass_grams": 307.0, "total_fat_grams": 16.9, "total_carb_grams": 36.0, "total_protein_grams": 32.0, "num_ingredients": 11, "ingredient_summary": "spinach (cooked) (3.4g), shallots (6.9g), salt (0.1g), onions (3.4g), broccoli (77.0g), olive oil (0.7g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562698276/rgb.png"}, {"dish_id": "dish_1561392593", "tier": "Medium-Low", "ground_truth_calories": 354.0, "total_mass_grams": 325.0, "total_fat_grams": 24.3, "total_carb_grams": 19.1, "total_protein_grams": 16.4, "num_ingredients": 11, "ingredient_summary": "vinegar (4.3g), olive oil (8.5g), broccoli (68.0g), berries (75.0g), bell peppers (10.7g), radishes (8.5g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561392593/rgb.png"}, {"dish_id": "dish_1566931586", "tier": "Medium-Low", "ground_truth_calories": 379.4, "total_mass_grams": 389.0, "total_fat_grams": 15.3, "total_carb_grams": 15.5, "total_protein_grams": 46.6, "num_ingredients": 4, "ingredient_summary": "olives (91.0g), mixed greens (91.0g), chicken (136.0g), broccoli (71.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566931586/rgb.png"}, {"dish_id": "dish_1563552016", "tier": "Medium-Low", "ground_truth_calories": 493.6, "total_mass_grams": 174.0, "total_fat_grams": 19.5, "total_carb_grams": 70.0, "total_protein_grams": 10.7, "num_ingredients": 2, "ingredient_summary": "syrup (38.1g), waffles (135.9g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563552016/rgb.png"}, {"dish_id": "dish_1558722247", "tier": "Medium-Low", "ground_truth_calories": 604.6, "total_mass_grams": 343.0, "total_fat_grams": 43.2, "total_carb_grams": 46.8, "total_protein_grams": 21.5, "num_ingredients": 3, "ingredient_summary": "almonds (86.0g), cauliflower (97.0g), apple (160.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558722247/rgb.png"}, {"dish_id": "dish_1561061687", "tier": "Medium-Low", "ground_truth_calories": 508.3, "total_mass_grams": 408.0, "total_fat_grams": 24.1, "total_carb_grams": 33.0, "total_protein_grams": 41.7, "num_ingredients": 26, "ingredient_summary": "shallots (6.2g), garlic (4.5g), croutons (0.3g), jalapenos (1.3g), onions (2.6g), cranberries (2.3g), +20 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561061687/rgb.png"}, {"dish_id": "dish_1565811091", "tier": "Medium-Low", "ground_truth_calories": 504.4, "total_mass_grams": 371.0, "total_fat_grams": 17.6, "total_carb_grams": 9.1, "total_protein_grams": 74.7, "num_ingredients": 5, "ingredient_summary": "broccoli (52.0g), spinach (raw) (20.0g), carrot (47.0g), chicken (205.0g), goat cheese (47.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565811091/rgb.png"}, {"dish_id": "dish_1566316587", "tier": "Medium-Low", "ground_truth_calories": 646.8, "total_mass_grams": 311.0, "total_fat_grams": 50.8, "total_carb_grams": 20.9, "total_protein_grams": 26.9, "num_ingredients": 11, "ingredient_summary": "olive oil (12.4g), vinegar (6.2g), radishes (12.4g), bacon (19.0g), garlic (0.8g), bell peppers (15.5g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566316587/rgb.png"}, {"dish_id": "dish_1560367715", "tier": "Medium-Low", "ground_truth_calories": 597.5, "total_mass_grams": 305.0, "total_fat_grams": 26.7, "total_carb_grams": 40.8, "total_protein_grams": 44.5, "num_ingredients": 16, "ingredient_summary": "broccoli (5.1g), brown rice (20.4g), vinegar (1.0g), pork (152.9g), olive oil (4.1g), lemon juice (1.0g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560367715/rgb.png"}, {"dish_id": "dish_1565195493", "tier": "Medium-Low", "ground_truth_calories": 407.0, "total_mass_grams": 291.0, "total_fat_grams": 21.0, "total_carb_grams": 20.2, "total_protein_grams": 31.0, "num_ingredients": 5, "ingredient_summary": "sweet potato (100.5g), chicken apple sausage (23.0g), bacon (38.0g), olive oil (2.5g), egg whites (127.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565195493/rgb.png"}, {"dish_id": "dish_1564159636", "tier": "Medium-Low", "ground_truth_calories": 487.9, "total_mass_grams": 134.0, "total_fat_grams": 37.4, "total_carb_grams": 5.3, "total_protein_grams": 31.4, "num_ingredients": 7, "ingredient_summary": "sausage (56.0g), olive oil (0.2g), pepper (0.0g), salt (0.1g), turkey bacon (15.0g), bacon (42.0g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564159636/rgb.png"}, {"dish_id": "dish_1560798960", "tier": "Medium-Low", "ground_truth_calories": 493.7, "total_mass_grams": 429.0, "total_fat_grams": 22.4, "total_carb_grams": 53.3, "total_protein_grams": 24.2, "num_ingredients": 24, "ingredient_summary": "cauliflower (90.0g), croutons (0.2g), sunflower seeds (1.6g), garlic (0.2g), onions (1.9g), tofu (81.0g), +18 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560798960/rgb.png"}, {"dish_id": "dish_1562618000", "tier": "Medium-High", "ground_truth_calories": 686.0, "total_mass_grams": 448.0, "total_fat_grams": 36.1, "total_carb_grams": 30.8, "total_protein_grams": 59.2, "num_ingredients": 16, "ingredient_summary": "vinegar (8.7g), olive oil (16.4g), garlic (0.5g), mustard greens (16.7g), pepper (6.7g), salt (0.7g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562618000/rgb.png"}, {"dish_id": "dish_1558720236", "tier": "Medium-High", "ground_truth_calories": 887.8, "total_mass_grams": 407.0, "total_fat_grams": 66.4, "total_carb_grams": 60.5, "total_protein_grams": 31.9, "num_ingredients": 4, "ingredient_summary": "apple (143.0g), carrot (74.0g), cauliflower (57.0g), almonds (133.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558720236/rgb.png"}, {"dish_id": "dish_1562963704", "tier": "Medium-High", "ground_truth_calories": 663.8, "total_mass_grams": 479.0, "total_fat_grams": 28.4, "total_carb_grams": 66.1, "total_protein_grams": 35.9, "num_ingredients": 19, "ingredient_summary": "radishes (3.5g), bok choy (12.3g), vinegar (3.5g), salt (0.6g), roasted potatoes (103.0g), mixed greens (17.5g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562963704/rgb.png"}, {"dish_id": "dish_1566591080", "tier": "Medium-High", "ground_truth_calories": 909.6, "total_mass_grams": 549.0, "total_fat_grams": 43.5, "total_carb_grams": 19.4, "total_protein_grams": 103.6, "num_ingredients": 11, "ingredient_summary": "pepper (8.6g), salmon (171.2g), chicken (142.0g), olives (14.0g), arugula (6.8g), brown rice (34.2g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566591080/rgb.png"}, {"dish_id": "dish_1560527191", "tier": "Medium-High", "ground_truth_calories": 664.0, "total_mass_grams": 318.0, "total_fat_grams": 45.4, "total_carb_grams": 28.9, "total_protein_grams": 34.4, "num_ingredients": 13, "ingredient_summary": "mustard greens (4.6g), cucumbers (3.7g), bacon (34.0g), berries (81.0g), tomatoes (2.3g), garlic (0.3g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560527191/rgb.png"}, {"dish_id": "dish_1568305257", "tier": "Medium-High", "ground_truth_calories": 837.1, "total_mass_grams": 420.0, "total_fat_grams": 65.5, "total_carb_grams": 9.3, "total_protein_grams": 52.1, "num_ingredients": 10, "ingredient_summary": "turkey bacon (82.0g), salt (0.4g), bell peppers (15.0g), mustard (0.8g), olive oil (12.0g), cucumbers (15.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568305257/rgb.png"}, {"dish_id": "dish_1558380557", "tier": "Medium-High", "ground_truth_calories": 699.3, "total_mass_grams": 212.0, "total_fat_grams": 54.4, "total_carb_grams": 25.0, "total_protein_grams": 32.7, "num_ingredients": 3, "ingredient_summary": "grapes (52.0g), almonds (48.0g), sausage (112.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558380557/rgb.png"}, {"dish_id": "dish_1564082462", "tier": "Medium-High", "ground_truth_calories": 905.4, "total_mass_grams": 609.0, "total_fat_grams": 32.1, "total_carb_grams": 40.5, "total_protein_grams": 108.8, "num_ingredients": 25, "ingredient_summary": "soy sauce (2.9g), pesto (10.7g), cherry tomatoes (6.7g), brown rice (27.5g), chard (6.9g), spinach (raw) (2.7g), +19 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564082462/rgb.png"}, {"dish_id": "dish_1565972715", "tier": "Medium-High", "ground_truth_calories": 657.5, "total_mass_grams": 510.0, "total_fat_grams": 46.8, "total_carb_grams": 15.7, "total_protein_grams": 43.2, "num_ingredients": 13, "ingredient_summary": "cucumbers (7.8g), vinegar (3.1g), green beans (81.9g), scrambled eggs (237.0g), pepper (0.2g), lettuce (15.6g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565972715/rgb.png"}, {"dish_id": "dish_1564686832", "tier": "Medium-High", "ground_truth_calories": 829.7, "total_mass_grams": 392.0, "total_fat_grams": 39.7, "total_carb_grams": 64.5, "total_protein_grams": 54.2, "num_ingredients": 14, "ingredient_summary": "onions (4.2g), brownies (49.0g), olives (25.0g), chicken breast (122.0g), salt (0.5g), lime (1.1g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564686832/rgb.png"}, {"dish_id": "dish_1566502573", "tier": "Medium-High", "ground_truth_calories": 821.5, "total_mass_grams": 487.0, "total_fat_grams": 43.2, "total_carb_grams": 60.7, "total_protein_grams": 51.2, "num_ingredients": 28, "ingredient_summary": "chive (0.0g), bok choy (14.9g), chicken (31.7g), white rice (14.9g), pork (104.2g), arugula (21.5g), +22 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566502573/rgb.png"}, {"dish_id": "dish_1561577573", "tier": "Medium-High", "ground_truth_calories": 755.4, "total_mass_grams": 346.0, "total_fat_grams": 40.6, "total_carb_grams": 25.0, "total_protein_grams": 66.1, "num_ingredients": 10, "ingredient_summary": "onions (7.6g), bok choy (7.6g), millet (30.2g), beef (226.8g), salt (0.3g), spinach (cooked) (7.6g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561577573/rgb.png"}, {"dish_id": "dish_1561752996", "tier": "Medium-High", "ground_truth_calories": 848.7, "total_mass_grams": 415.0, "total_fat_grams": 42.9, "total_carb_grams": 36.3, "total_protein_grams": 76.8, "num_ingredients": 15, "ingredient_summary": "garlic (1.6g), arugula (3.5g), brown rice (17.4g), thyme (0.1g), salt (0.6g), cilantro (1.6g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561752996/rgb.png"}, {"dish_id": "dish_1561753075", "tier": "Medium-High", "ground_truth_calories": 945.7, "total_mass_grams": 668.0, "total_fat_grams": 47.4, "total_carb_grams": 46.3, "total_protein_grams": 82.7, "num_ingredients": 17, "ingredient_summary": "garlic (1.6g), salmon (87.1g), chicken thighs (162.6g), onions (0.4g), olive oil (5.1g), chili (1.4g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561753075/rgb.png"}, {"dish_id": "dish_1568401302", "tier": "Medium-High", "ground_truth_calories": 716.5, "total_mass_grams": 561.0, "total_fat_grams": 28.9, "total_carb_grams": 69.9, "total_protein_grams": 44.7, "num_ingredients": 7, "ingredient_summary": "fried rice (118.0g), salt (0.4g), squash (113.4g), carrot (75.0g), avocado (25.0g), olive oil (1.1g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568401302/rgb.png"}, {"dish_id": "dish_1563307575", "tier": "Medium-High", "ground_truth_calories": 671.1, "total_mass_grams": 419.0, "total_fat_grams": 38.1, "total_carb_grams": 25.1, "total_protein_grams": 53.9, "num_ingredients": 19, "ingredient_summary": "orange juice (1.3g), onions (9.3g), garlic (1.3g), soy sauce (1.3g), green onions (1.3g), salt (0.5g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563307575/rgb.png"}, {"dish_id": "dish_1561739805", "tier": "Medium-High", "ground_truth_calories": 696.4, "total_mass_grams": 1102.0, "total_fat_grams": 23.3, "total_carb_grams": 101.3, "total_protein_grams": 29.1, "num_ingredients": 10, "ingredient_summary": "raspberries (2.6g), broccoli (264.0g), berries (249.0g), sweet potato (84.0g), scrambled eggs (165.0g), cantaloupe (300.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561739805/rgb.png"}, {"dish_id": "dish_1563910639", "tier": "Medium-High", "ground_truth_calories": 697.3, "total_mass_grams": 447.0, "total_fat_grams": 29.1, "total_carb_grams": 20.5, "total_protein_grams": 81.0, "num_ingredients": 14, "ingredient_summary": "garlic (9.7g), lime (0.4g), chili (0.8g), salt (0.8g), onions (6.5g), lemon juice (3.2g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563910639/rgb.png"}, {"dish_id": "dish_1559244649", "tier": "Medium-High", "ground_truth_calories": 906.8, "total_mass_grams": 455.0, "total_fat_grams": 40.3, "total_carb_grams": 51.7, "total_protein_grams": 83.3, "num_ingredients": 20, "ingredient_summary": "pepper (3.3g), chive (0.3g), olive oil (16.4g), mustard greens (8.0g), walnuts (1.4g), basil (0.3g), +14 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1559244649/rgb.png"}, {"dish_id": "dish_1560799647", "tier": "Medium-High", "ground_truth_calories": 723.4, "total_mass_grams": 474.0, "total_fat_grams": 39.7, "total_carb_grams": 39.6, "total_protein_grams": 50.0, "num_ingredients": 17, "ingredient_summary": "garlic (1.0g), spinach (raw) (56.2g), squash (118.4g), pork (146.4g), mustard (1.1g), pizza (74.0g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560799647/rgb.png"}, {"dish_id": "dish_1563998513", "tier": "Medium-High", "ground_truth_calories": 757.5, "total_mass_grams": 599.0, "total_fat_grams": 38.7, "total_carb_grams": 51.6, "total_protein_grams": 54.2, "num_ingredients": 27, "ingredient_summary": "chicken (70.9g), pepper (0.4g), cookies (25.0g), brown rice (18.4g), arugula (1.8g), vinegar (0.2g), +21 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563998513/rgb.png"}, {"dish_id": "dish_1564169958", "tier": "Medium-High", "ground_truth_calories": 787.8, "total_mass_grams": 563.0, "total_fat_grams": 37.7, "total_carb_grams": 46.7, "total_protein_grams": 67.1, "num_ingredients": 19, "ingredient_summary": "chili (1.6g), tomatoes (18.5g), cilantro (1.9g), olive oil (0.9g), parsley (0.7g), cucumbers (19.1g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564169958/rgb.png"}, {"dish_id": "dish_1566590056", "tier": "Medium-High", "ground_truth_calories": 815.6, "total_mass_grams": 519.0, "total_fat_grams": 34.7, "total_carb_grams": 63.9, "total_protein_grams": 62.9, "num_ingredients": 19, "ingredient_summary": "tomatoes (14.2g), pepper (0.5g), millet (28.5g), vinegar (2.7g), tofu (122.0g), jalapenos (5.7g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566590056/rgb.png"}, {"dish_id": "dish_1562790321", "tier": "Medium-High", "ground_truth_calories": 767.5, "total_mass_grams": 354.0, "total_fat_grams": 53.0, "total_carb_grams": 39.8, "total_protein_grams": 37.5, "num_ingredients": 12, "ingredient_summary": "salt (1.0g), pepper (0.1g), walnuts (34.0g), olive oil (8.7g), pizza (86.0g), chard (10.1g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562790321/rgb.png"}, {"dish_id": "dish_1566328776", "tier": "Medium-High", "ground_truth_calories": 901.8, "total_mass_grams": 436.0, "total_fat_grams": 28.7, "total_carb_grams": 78.3, "total_protein_grams": 77.7, "num_ingredients": 3, "ingredient_summary": "chicken (167.0g), cherry tomatoes (36.0g), pizza (233.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566328776/rgb.png"}, {"dish_id": "dish_1566328831", "tier": "Medium-High", "ground_truth_calories": 941.6, "total_mass_grams": 500.0, "total_fat_grams": 30.0, "total_carb_grams": 85.8, "total_protein_grams": 78.1, "num_ingredients": 5, "ingredient_summary": "pizza (233.0g), cherry tomatoes (36.0g), chicken (167.0g), pineapple (52.0g), olives (12.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566328831/rgb.png"}, {"dish_id": "dish_1566590007", "tier": "Medium-High", "ground_truth_calories": 740.8, "total_mass_grams": 457.0, "total_fat_grams": 29.2, "total_carb_grams": 58.0, "total_protein_grams": 61.8, "num_ingredients": 13, "ingredient_summary": "bell peppers (14.2g), garlic (2.8g), tomatoes (14.2g), salt (2.8g), onions (5.7g), olive oil (5.7g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566590007/rgb.png"}, {"dish_id": "dish_1560367733", "tier": "Medium-High", "ground_truth_calories": 661.4, "total_mass_grams": 388.0, "total_fat_grams": 26.8, "total_carb_grams": 54.9, "total_protein_grams": 46.2, "num_ingredients": 17, "ingredient_summary": "vinegar (1.0g), cherry tomatoes (5.1g), white rice (97.0g), croutons (10.0g), garlic (2.0g), broccoli (5.1g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560367733/rgb.png"}, {"dish_id": "dish_1566328805", "tier": "Medium-High", "ground_truth_calories": 927.8, "total_mass_grams": 488.0, "total_fat_grams": 28.7, "total_carb_grams": 85.1, "total_protein_grams": 78.0, "num_ingredients": 4, "ingredient_summary": "cherry tomatoes (36.0g), pizza (233.0g), chicken (167.0g), pineapple (52.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566328805/rgb.png"}, {"dish_id": "dish_1562789328", "tier": "Medium-High", "ground_truth_calories": 656.6, "total_mass_grams": 503.0, "total_fat_grams": 27.8, "total_carb_grams": 69.7, "total_protein_grams": 40.7, "num_ingredients": 18, "ingredient_summary": "broccoli (2.5g), tofu (68.0g), corn on the cob (196.0g), vinegar (0.5g), lemon juice (0.5g), shallots (0.5g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562789328/rgb.png"}, {"dish_id": "dish_1561580659", "tier": "Medium-High", "ground_truth_calories": 754.4, "total_mass_grams": 480.0, "total_fat_grams": 43.8, "total_carb_grams": 16.3, "total_protein_grams": 68.9, "num_ingredients": 17, "ingredient_summary": "salt (0.3g), olive oil (1.6g), cherry tomatoes (8.0g), beef (247.4g), feta cheese (3.2g), cauliflower (81.0g), +11 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561580659/rgb.png"}, {"dish_id": "dish_1563910676", "tier": "Medium-High", "ground_truth_calories": 758.0, "total_mass_grams": 501.0, "total_fat_grams": 34.8, "total_carb_grams": 22.8, "total_protein_grams": 82.1, "num_ingredients": 18, "ingredient_summary": "brown rice (64.8g), ginger (1.6g), radishes (5.5g), mustard (0.3g), shallots (9.7g), lime (0.4g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563910676/rgb.png"}, {"dish_id": "dish_1565974409", "tier": "Medium-High", "ground_truth_calories": 688.6, "total_mass_grams": 228.0, "total_fat_grams": 50.0, "total_carb_grams": 5.5, "total_protein_grams": 51.5, "num_ingredients": 2, "ingredient_summary": "chicken apple sausage (143.0g), bacon (85.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565974409/rgb.png"}, {"dish_id": "dish_1563910708", "tier": "Medium-High", "ground_truth_calories": 786.2, "total_mass_grams": 555.0, "total_fat_grams": 36.2, "total_carb_grams": 26.5, "total_protein_grams": 83.0, "num_ingredients": 19, "ingredient_summary": "blueberries (5.5g), vinegar (2.7g), lemon juice (3.2g), mustard (0.3g), olive oil (9.2g), green beans (52.7g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563910708/rgb.png"}, {"dish_id": "dish_1564601638", "tier": "Medium-High", "ground_truth_calories": 712.5, "total_mass_grams": 541.0, "total_fat_grams": 46.9, "total_carb_grams": 43.0, "total_protein_grams": 51.1, "num_ingredients": 21, "ingredient_summary": "olive oil (4.3g), kale (3.8g), cherry tomatoes (3.0g), chard (3.0g), turnips (73.0g), salt (1.4g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564601638/rgb.png"}, {"dish_id": "dish_1563309398", "tier": "Medium-High", "ground_truth_calories": 656.0, "total_mass_grams": 604.0, "total_fat_grams": 20.2, "total_carb_grams": 23.3, "total_protein_grams": 93.2, "num_ingredients": 15, "ingredient_summary": "salt (0.4g), spinach (raw) (7.1g), lemon juice (1.4g), olives (33.0g), bell peppers (29.0g), lettuce (66.0g), +9 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563309398/rgb.png"}, {"dish_id": "dish_1561737271", "tier": "Medium-High", "ground_truth_calories": 762.4, "total_mass_grams": 520.0, "total_fat_grams": 45.8, "total_carb_grams": 46.5, "total_protein_grams": 41.0, "num_ingredients": 5, "ingredient_summary": "sausage (117.0g), hash browns (60.0g), pineapple (126.0g), broccoli (72.0g), egg whites (145.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1561737271/rgb.png"}, {"dish_id": "dish_1563826306", "tier": "Medium-High", "ground_truth_calories": 882.8, "total_mass_grams": 312.0, "total_fat_grams": 48.9, "total_carb_grams": 69.6, "total_protein_grams": 44.4, "num_ingredients": 9, "ingredient_summary": "garlic (1.4g), lime (1.4g), salt (0.7g), brownies (51.0g), olive oil (0.2g), wheat berry (20.3g), +3 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563826306/rgb.png"}, {"dish_id": "dish_1563480534", "tier": "Medium-High", "ground_truth_calories": 797.6, "total_mass_grams": 576.0, "total_fat_grams": 33.9, "total_carb_grams": 35.5, "total_protein_grams": 81.9, "num_ingredients": 23, "ingredient_summary": "garlic (8.5g), wine (1.3g), lemon juice (2.4g), onions (11.1g), jalapenos (2.4g), brown rice (48.2g), +17 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563480534/rgb.png"}, {"dish_id": "dish_1565640365", "tier": "Medium-High", "ground_truth_calories": 720.9, "total_mass_grams": 623.0, "total_fat_grams": 22.4, "total_carb_grams": 54.6, "total_protein_grams": 75.1, "num_ingredients": 11, "ingredient_summary": "wheat berry (17.9g), olive oil (0.7g), thyme (0.4g), salt (0.2g), bell peppers (54.0g), garlic (1.8g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565640365/rgb.png"}, {"dish_id": "dish_1562010315", "tier": "Medium-High", "ground_truth_calories": 671.1, "total_mass_grams": 477.0, "total_fat_grams": 38.2, "total_carb_grams": 19.7, "total_protein_grams": 61.9, "num_ingredients": 14, "ingredient_summary": "caesar salad (120.0g), jalapenos (0.3g), pork (206.8g), pepper (0.8g), wheat berry (27.6g), mayonnaise (5.5g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562010315/rgb.png"}, {"dish_id": "dish_1563827482", "tier": "Medium-High", "ground_truth_calories": 894.5, "total_mass_grams": 621.0, "total_fat_grams": 56.0, "total_carb_grams": 24.5, "total_protein_grams": 77.4, "num_ingredients": 11, "ingredient_summary": "caesar salad (138.0g), garlic (3.6g), broccoli (36.7g), zucchini (36.7g), lime (3.6g), olive oil (0.5g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563827482/rgb.png"}, {"dish_id": "dish_1565203094", "tier": "Medium-High", "ground_truth_calories": 898.0, "total_mass_grams": 485.0, "total_fat_grams": 27.7, "total_carb_grams": 38.0, "total_protein_grams": 116.7, "num_ingredients": 12, "ingredient_summary": "rosemary (0.4g), parsley (0.4g), garlic (1.7g), chicken (341.4g), pizza (88.0g), salt (1.7g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565203094/rgb.png"}, {"dish_id": "dish_1558373492", "tier": "Medium-High", "ground_truth_calories": 888.2, "total_mass_grams": 325.0, "total_fat_grams": 66.8, "total_carb_grams": 42.3, "total_protein_grams": 39.0, "num_ingredients": 4, "ingredient_summary": "cantaloupe (40.0g), almonds (72.0g), sausage (113.0g), grapes (100.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558373492/rgb.png"}, {"dish_id": "dish_1566316612", "tier": "Medium-High", "ground_truth_calories": 665.2, "total_mass_grams": 367.0, "total_fat_grams": 51.5, "total_carb_grams": 24.2, "total_protein_grams": 27.5, "num_ingredients": 12, "ingredient_summary": "hash browns (48.0g), bacon (19.0g), vinegar (6.2g), radishes (12.4g), scrambled eggs (180.0g), garlic (0.8g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566316612/rgb.png"}, {"dish_id": "dish_1567537280", "tier": "Medium-High", "ground_truth_calories": 733.0, "total_mass_grams": 404.0, "total_fat_grams": 34.7, "total_carb_grams": 51.1, "total_protein_grams": 51.3, "num_ingredients": 16, "ingredient_summary": "chili (0.9g), onions (1.9g), kale (4.7g), orange juice (0.9g), soy sauce (0.9g), green onions (0.9g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567537280/rgb.png"}, {"dish_id": "dish_1565203015", "tier": "Medium-High", "ground_truth_calories": 663.9, "total_mass_grams": 397.0, "total_fat_grams": 19.1, "total_carb_grams": 8.9, "total_protein_grams": 107.1, "num_ingredients": 11, "ingredient_summary": "parsley (0.4g), chard (8.5g), lemon juice (1.3g), garlic (1.7g), chicken (341.4g), chive (0.4g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565203015/rgb.png"}, {"dish_id": "dish_1562180089", "tier": "Medium-High", "ground_truth_calories": 851.3, "total_mass_grams": 437.0, "total_fat_grams": 57.8, "total_carb_grams": 19.0, "total_protein_grams": 60.7, "num_ingredients": 12, "ingredient_summary": "vegetable oil (34.9g), olive oil (1.8g), buttermilk (27.9g), flour (14.0g), chicken (139.7g), bacon (36.0g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562180089/rgb.png"}, {"dish_id": "dish_1565811139", "tier": "Medium-High", "ground_truth_calories": 902.2, "total_mass_grams": 416.0, "total_fat_grams": 62.6, "total_carb_grams": 9.1, "total_protein_grams": 74.7, "num_ingredients": 6, "ingredient_summary": "carrot (47.0g), goat cheese (47.0g), olive oil (45.0g), broccoli (52.0g), spinach (raw) (20.0g), chicken (205.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565811139/rgb.png"}, {"dish_id": "dish_1566499686", "tier": "Medium-High", "ground_truth_calories": 674.7, "total_mass_grams": 388.0, "total_fat_grams": 37.7, "total_carb_grams": 35.9, "total_protein_grams": 48.4, "num_ingredients": 16, "ingredient_summary": "pepper (0.6g), spinach (raw) (61.3g), vinegar (4.9g), white rice (23.8g), olives (37.0g), bok choy (23.8g), +10 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566499686/rgb.png"}, {"dish_id": "dish_1566414723", "tier": "Medium-High", "ground_truth_calories": 728.8, "total_mass_grams": 482.0, "total_fat_grams": 31.7, "total_carb_grams": 24.9, "total_protein_grams": 98.0, "num_ingredients": 22, "ingredient_summary": "green onions (0.6g), salt (2.2g), white wine (0.5g), olive oil (3.2g), rosemary (0.6g), lemon juice (0.5g), +16 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566414723/rgb.png"}, {"dish_id": "dish_1563568357", "tier": "Medium-High", "ground_truth_calories": 767.7, "total_mass_grams": 448.0, "total_fat_grams": 38.1, "total_carb_grams": 69.4, "total_protein_grams": 42.9, "num_ingredients": 23, "ingredient_summary": "cheese pizza (45.0g), white rice (14.1g), wheat berry (23.3g), ground turkey (106.0g), parsley (3.9g), arugula (9.6g), +17 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563568357/rgb.png"}, {"dish_id": "dish_1558378634", "tier": "Medium-High", "ground_truth_calories": 875.0, "total_mass_grams": 315.0, "total_fat_grams": 64.8, "total_carb_grams": 44.3, "total_protein_grams": 38.0, "num_ingredients": 3, "ingredient_summary": "grapes (134.0g), almonds (68.0g), sausage (113.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558378634/rgb.png"}, {"dish_id": "dish_1563480505", "tier": "Medium-High", "ground_truth_calories": 767.3, "total_mass_grams": 475.0, "total_fat_grams": 33.5, "total_carb_grams": 29.5, "total_protein_grams": 79.8, "num_ingredients": 21, "ingredient_summary": "salmon (240.8g), mushroom (9.4g), spinach (raw) (18.3g), brown rice (48.2g), cream (2.5g), olive oil (2.3g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563480505/rgb.png"}, {"dish_id": "dish_1562790855", "tier": "Medium-High", "ground_truth_calories": 739.4, "total_mass_grams": 507.0, "total_fat_grams": 67.9, "total_carb_grams": 23.8, "total_protein_grams": 78.4, "num_ingredients": 21, "ingredient_summary": "radishes (7.8g), fish (256.8g), cherry tomatoes (3.6g), shallots (0.7g), onions (11.7g), mustard (0.5g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562790855/rgb.png"}, {"dish_id": "dish_1558630325", "tier": "Medium-High", "ground_truth_calories": 751.5, "total_mass_grams": 269.0, "total_fat_grams": 58.1, "total_carb_grams": 46.6, "total_protein_grams": 27.1, "num_ingredients": 3, "ingredient_summary": "apple (142.0g), garden salad (11.0g), almonds (116.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558630325/rgb.png"}, {"dish_id": "dish_1558721827", "tier": "Medium-High", "ground_truth_calories": 886.5, "total_mass_grams": 429.0, "total_fat_grams": 66.5, "total_carb_grams": 59.5, "total_protein_grams": 32.4, "num_ingredients": 4, "ingredient_summary": "cantaloupe (56.0g), apple (143.0g), cauliflower (97.0g), almonds (133.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558721827/rgb.png"}, {"dish_id": "dish_1567789256", "tier": "Medium-High", "ground_truth_calories": 792.2, "total_mass_grams": 319.0, "total_fat_grams": 64.7, "total_carb_grams": 9.4, "total_protein_grams": 41.4, "num_ingredients": 11, "ingredient_summary": "salt (0.4g), garlic (0.8g), bell peppers (16.0g), scrambled eggs (123.0g), bacon (19.0g), cucumbers (16.0g), +5 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1567789256/rgb.png"}, {"dish_id": "dish_1566329021", "tier": "Medium-High", "ground_truth_calories": 792.7, "total_mass_grams": 486.0, "total_fat_grams": 28.0, "total_carb_grams": 74.7, "total_protein_grams": 59.3, "num_ingredients": 19, "ingredient_summary": "bell peppers (10.2g), shallots (3.9g), beef (58.0g), garlic (2.0g), vinegar (0.1g), spinach (raw) (4.1g), +13 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566329021/rgb.png"}, {"dish_id": "dish_1562963325", "tier": "Medium-High", "ground_truth_calories": 760.2, "total_mass_grams": 379.0, "total_fat_grams": 31.5, "total_carb_grams": 65.3, "total_protein_grams": 51.8, "num_ingredients": 10, "ingredient_summary": "sugar (14.2g), garlic (4.7g), pepper (0.5g), white rice (19.0g), bok choy (19.0g), pork (133.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562963325/rgb.png"}, {"dish_id": "dish_1563480608", "tier": "Medium-High", "ground_truth_calories": 886.9, "total_mass_grams": 650.0, "total_fat_grams": 40.4, "total_carb_grams": 42.6, "total_protein_grams": 83.2, "num_ingredients": 29, "ingredient_summary": "beef (62.7g), salt (0.9g), cilantro (2.4g), olive oil (8.7g), cucumbers (8.0g), jalapenos (2.4g), +23 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563480608/rgb.png"}, {"dish_id": "dish_1566591048", "tier": "Medium-High", "ground_truth_calories": 876.2, "total_mass_grams": 481.0, "total_fat_grams": 42.7, "total_carb_grams": 12.7, "total_protein_grams": 103.0, "num_ingredients": 10, "ingredient_summary": "olives (14.0g), brown rice (34.2g), arugula (6.8g), chicken salad (97.0g), pepper (8.6g), chicken (142.0g), +4 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566591048/rgb.png"}, {"dish_id": "dish_1568060746", "tier": "Medium-High", "ground_truth_calories": 778.8, "total_mass_grams": 395.0, "total_fat_grams": 53.0, "total_carb_grams": 12.2, "total_protein_grams": 64.4, "num_ingredients": 5, "ingredient_summary": "olive oil (36.0g), mixed greens (61.0g), avocado (64.0g), onions (36.0g), chicken (198.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568060746/rgb.png"}, {"dish_id": "dish_1558723894", "tier": "Medium-High", "ground_truth_calories": 757.6, "total_mass_grams": 405.0, "total_fat_grams": 50.6, "total_carb_grams": 68.3, "total_protein_grams": 26.3, "num_ingredients": 4, "ingredient_summary": "almonds (96.0g), cauliflower (59.0g), corn on the cob (107.0g), apple (143.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558723894/rgb.png"}, {"dish_id": "dish_1565118096", "tier": "Medium-High", "ground_truth_calories": 656.4, "total_mass_grams": 588.0, "total_fat_grams": 13.0, "total_carb_grams": 50.0, "total_protein_grams": 82.4, "num_ingredients": 7, "ingredient_summary": "chicken breast (133.0g), tuna (102.0g), cheese pizza (76.0g), cherry tomatoes (73.0g), bell peppers (69.0g), carrot (69.0g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565118096/rgb.png"}, {"dish_id": "dish_1558379876", "tier": "High", "ground_truth_calories": 1148.3, "total_mass_grams": 374.0, "total_fat_grams": 89.8, "total_carb_grams": 32.1, "total_protein_grams": 68.3, "num_ingredients": 3, "ingredient_summary": "deprecated (143.0g), sausage (112.0g), almonds (119.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558379876/rgb.png"}, {"dish_id": "dish_1564170010", "tier": "High", "ground_truth_calories": 1264.5, "total_mass_grams": 831.0, "total_fat_grams": 61.9, "total_carb_grams": 57.3, "total_protein_grams": 116.9, "num_ingredients": 23, "ingredient_summary": "tortilla (74.2g), cucumbers (19.1g), tatsoi (76.4g), kale (19.1g), cilantro (1.9g), tomatoes (18.5g), +17 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564170010/rgb.png"}, {"dish_id": "dish_1558373159", "tier": "High", "ground_truth_calories": 1013.3, "total_mass_grams": 334.0, "total_fat_grams": 79.1, "total_carb_grams": 30.5, "total_protein_grams": 60.6, "num_ingredients": 3, "ingredient_summary": "sausage (73.0g), deprecated (142.0g), almonds (119.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558373159/rgb.png"}, {"dish_id": "dish_1563567372", "tier": "High", "ground_truth_calories": 951.9, "total_mass_grams": 590.0, "total_fat_grams": 66.2, "total_carb_grams": 81.6, "total_protein_grams": 75.4, "num_ingredients": 13, "ingredient_summary": "cherry tomatoes (10.7g), fish (268.0g), salt (0.7g), wheat berry (53.6g), lemon juice (5.4g), chard (10.7g), +7 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563567372/rgb.png"}, {"dish_id": "dish_1563476551", "tier": "High", "ground_truth_calories": 1116.4, "total_mass_grams": 750.0, "total_fat_grams": 51.4, "total_carb_grams": 74.7, "total_protein_grams": 85.5, "num_ingredients": 26, "ingredient_summary": "shallots (6.3g), salmon (208.9g), mushroom (11.4g), ginger (1.0g), squash (69.3g), brownies (51.0g), +20 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563476551/rgb.png"}, {"dish_id": "dish_1565033265", "tier": "High", "ground_truth_calories": 1324.1, "total_mass_grams": 625.0, "total_fat_grams": 106.3, "total_carb_grams": 19.2, "total_protein_grams": 76.7, "num_ingredients": 6, "ingredient_summary": "olives (127.0g), mixed greens (55.0g), broccoli (92.0g), chicken (233.0g), vinaigrette (71.0g), olive oil (47.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1565033265/rgb.png"}, {"dish_id": "dish_1566501153", "tier": "High", "ground_truth_calories": 1006.9, "total_mass_grams": 419.0, "total_fat_grams": 83.8, "total_carb_grams": 17.9, "total_protein_grams": 50.9, "num_ingredients": 7, "ingredient_summary": "olives (97.0g), carrot (38.0g), spinach (raw) (25.0g), parmesan cheese (34.0g), chicken (112.0g), avocado (63.0g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566501153/rgb.png"}, {"dish_id": "dish_1560356627", "tier": "High", "ground_truth_calories": 980.0, "total_mass_grams": 301.0, "total_fat_grams": 76.0, "total_carb_grams": 6.9, "total_protein_grams": 63.1, "num_ingredients": 3, "ingredient_summary": "scrambled eggs (121.0g), bacon (92.0g), sausage (88.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1560356627/rgb.png"}, {"dish_id": "dish_1563478751", "tier": "High", "ground_truth_calories": 950.4, "total_mass_grams": 545.0, "total_fat_grams": 45.3, "total_carb_grams": 54.1, "total_protein_grams": 74.8, "num_ingredients": 12, "ingredient_summary": "cream (9.6g), mushroom (35.8g), spinach (raw) (23.9g), olive oil (1.8g), pasta (191.1g), beef (238.9g), +6 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563478751/rgb.png"}, {"dish_id": "dish_1564686855", "tier": "High", "ground_truth_calories": 1030.2, "total_mass_grams": 435.0, "total_fat_grams": 52.1, "total_carb_grams": 85.9, "total_protein_grams": 56.8, "num_ingredients": 14, "ingredient_summary": "cucumbers (10.6g), onions (4.2g), brownies (92.0g), chicken breast (122.0g), cherry tomatoes (8.5g), olives (25.0g), +8 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1564686855/rgb.png"}, {"dish_id": "dish_1563476463", "tier": "High", "ground_truth_calories": 1067.5, "total_mass_grams": 542.0, "total_fat_grams": 50.7, "total_carb_grams": 65.6, "total_protein_grams": 82.4, "num_ingredients": 23, "ingredient_summary": "garlic (7.8g), lime (0.3g), chili (0.5g), lemon juice (2.1g), brownies (51.0g), olive oil (2.1g), +17 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563476463/rgb.png"}, {"dish_id": "dish_1566931674", "tier": "High", "ground_truth_calories": 991.0, "total_mass_grams": 531.0, "total_fat_grams": 76.8, "total_carb_grams": 23.1, "total_protein_grams": 58.4, "num_ingredients": 7, "ingredient_summary": "chicken (136.0g), mixed greens (91.0g), avocado (72.0g), broccoli (71.0g), parmesan cheese (27.0g), olive oil (43.0g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1566931674/rgb.png"}, {"dish_id": "dish_1562180157", "tier": "High", "ground_truth_calories": 1009.2, "total_mass_grams": 557.0, "total_fat_grams": 72.2, "total_carb_grams": 25.5, "total_protein_grams": 63.8, "num_ingredients": 18, "ingredient_summary": "olive oil (14.2g), almonds (3.1g), spinach (raw) (77.7g), mustard (0.8g), bacon (36.0g), chicken (139.7g), +12 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562180157/rgb.png"}, {"dish_id": "dish_1563476577", "tier": "High", "ground_truth_calories": 1238.0, "total_mass_grams": 857.0, "total_fat_grams": 62.6, "total_carb_grams": 79.6, "total_protein_grams": 87.7, "num_ingredients": 30, "ingredient_summary": "beef (76.3g), broccoli (69.3g), shallots (6.3g), thyme (0.2g), onions (22.6g), ginger (1.0g), +24 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563476577/rgb.png"}, {"dish_id": "dish_1563305047", "tier": "High", "ground_truth_calories": 1032.3, "total_mass_grams": 806.0, "total_fat_grams": 36.7, "total_carb_grams": 28.7, "total_protein_grams": 143.5, "num_ingredients": 24, "ingredient_summary": "salt (1.8g), broccoli (3.2g), parsley (3.8g), lemon juice (1.9g), kale (1.6g), green onions (0.3g), +18 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563305047/rgb.png"}, {"dish_id": "dish_1563476408", "tier": "High", "ground_truth_calories": 990.4, "total_mass_grams": 513.0, "total_fat_grams": 47.9, "total_carb_grams": 55.9, "total_protein_grams": 79.2, "num_ingredients": 22, "ingredient_summary": "salmon (208.9g), garlic (7.8g), shallots (6.3g), chili (0.5g), lemon juice (2.1g), onions (11.8g), +16 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563476408/rgb.png"}, {"dish_id": "dish_1563304958", "tier": "High", "ground_truth_calories": 951.8, "total_mass_grams": 614.0, "total_fat_grams": 34.4, "total_carb_grams": 13.3, "total_protein_grams": 141.7, "num_ingredients": 21, "ingredient_summary": "garlic (2.2g), shallots (0.1g), beef (47.8g), white rice (6.4g), orange juice (0.3g), spinach (raw) (9.5g), +15 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563304958/rgb.png"}, {"dish_id": "dish_1562603867", "tier": "High", "ground_truth_calories": 1049.1, "total_mass_grams": 369.0, "total_fat_grams": 74.5, "total_carb_grams": 66.3, "total_protein_grams": 32.2, "num_ingredients": 7, "ingredient_summary": "mixed greens (58.7g), hash browns (146.0g), raspberries (5.9g), blackberries (5.9g), sausage (132.0g), vinaigrette (5.9g), +1 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562603867/rgb.png"}, {"dish_id": "dish_1563305005", "tier": "High", "ground_truth_calories": 1003.9, "total_mass_grams": 720.0, "total_fat_grams": 35.7, "total_carb_grams": 23.7, "total_protein_grams": 142.6, "num_ingredients": 23, "ingredient_summary": "carrot (104.5g), beef (47.8g), chicken (381.5g), wheat berry (38.2g), tofu (106.0g), spinach (raw) (9.5g), +17 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563305005/rgb.png"}, {"dish_id": "dish_1558375886", "tier": "High", "ground_truth_calories": 1050.5, "total_mass_grams": 277.0, "total_fat_grams": 84.2, "total_carb_grams": 42.8, "total_protein_grams": 44.9, "num_ingredients": 3, "ingredient_summary": "almonds (119.0g), sausage (92.0g), grapes (66.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1558375886/rgb.png"}, {"dish_id": "dish_1562603895", "tier": "High", "ground_truth_calories": 1268.2, "total_mass_grams": 517.0, "total_fat_grams": 90.8, "total_carb_grams": 68.6, "total_protein_grams": 47.0, "num_ingredients": 8, "ingredient_summary": "blackberries (5.9g), mixed greens (58.7g), vinaigrette (5.9g), hash browns (146.0g), scrambled eggs (148.0g), raspberries (5.9g), +2 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1562603895/rgb.png"}, {"dish_id": "dish_1563305083", "tier": "High", "ground_truth_calories": 1234.9, "total_mass_grams": 901.0, "total_fat_grams": 56.8, "total_carb_grams": 32.1, "total_protein_grams": 147.5, "num_ingredients": 29, "ingredient_summary": "chili (0.3g), lettuce (42.0g), shallots (0.1g), broccoli (3.2g), olive oil (22.9g), parsley (3.8g), +23 more", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1563305083/rgb.png"}, {"dish_id": "dish_1568318480", "tier": "High", "ground_truth_calories": 1119.3, "total_mass_grams": 587.0, "total_fat_grams": 67.2, "total_carb_grams": 9.2, "total_protein_grams": 117.9, "num_ingredients": 5, "ingredient_summary": "mixed greens (115.0g), chicken (222.0g), olive oil (24.0g), steak (182.0g), broccoli (44.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568318480/rgb.png"}, {"dish_id": "dish_1568401090", "tier": "High", "ground_truth_calories": 1159.0, "total_mass_grams": 614.0, "total_fat_grams": 73.2, "total_carb_grams": 18.4, "total_protein_grams": 108.7, "num_ingredients": 5, "ingredient_summary": "broccoli (71.0g), mixed greens (42.0g), avocado (125.0g), olive oil (42.0g), chicken (334.0g)", "image_url": "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568401090/rgb.png"}]')
df = pd.DataFrame(records)
df['image_path'] = df['dish_id'].apply(lambda x: f'data/raw_images/{x}.png')

In [ ]:
from PIL import Image
import requests, re
from tqdm import tqdm

df['model_predicted_calories'] = None
df['model_reasoning'] = None
df['error_kcal'] = None
df['percentage_error'] = None

PROMPT = '''You are an expert nutritionist assisting a human task performer.
Look at this overhead photograph of a plated meal.
1. Identify the food items and estimate their portion in grams.
2. Estimate the total calories (kcal) for the entire meal.
Respond in JSON format: {"breakdown": [{"item": "chicken", "grams": 150, "calories": 250}], "total_calories": 550, "reasoning": "..."}'''

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Estimating Calories on GPU'):
    dish_id = row['dish_id']
    actual_cals = float(row['ground_truth_calories'])
    img_url = row['image_url']
    
    try:
        resp = requests.get(img_url, stream=True, timeout=10)
        img = Image.open(resp.raw).convert('RGB')
        
        messages = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], padding=True, return_tensors='pt').to('cuda')
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=180)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        out_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        
        match = re.search(r'"total_calories"\s*:\s*([0-9.]+)', out_text)
        if match:
            pred_cals = float(match.group(1))
        else:
            nums = re.findall(r'\b[0-9]{2,4}\b', out_text)
            pred_cals = float(nums[-1]) if nums else 450.0
        
        pred_cals = max(50.0, min(1800.0, pred_cals))
        err = abs(pred_cals - actual_cals)
        pct = (err / actual_cals) * 100.0
        
        df.at[idx, 'model_predicted_calories'] = round(pred_cals, 1)
        df.at[idx, 'model_reasoning'] = out_text[:200]
        df.at[idx, 'error_kcal'] = round(err, 1)
        df.at[idx, 'percentage_error'] = round(pct, 1)
        
    except Exception as e:
        print(f'Error processing {dish_id}: {e}')

df.to_csv('study_dataset_predictions.csv', index=False)

In [ ]:
import numpy as np
valid = df.dropna(subset=['model_predicted_calories']).copy()
mae = float(valid['error_kcal'].astype(float).mean())
mape = float(valid['percentage_error'].astype(float).mean())
r = float(valid['ground_truth_calories'].astype(float).corr(valid['model_predicted_calories'].astype(float)))

print(f'Total Evaluated Dishes: {len(valid)}')
print(f'Mean Absolute Error (MAE): {mae:.1f} kcal')
print(f'Mean Absolute Pct Error (MAPE): {mape:.1f}%')
print(f'Pearson Correlation (r): {r:.3f}')

In [ ]:
from google.colab import files
files.download('study_dataset_predictions.csv')